# Agreement Experiments

This notebook tracks the current agreement experiment as compact batch outputs arrive. It joins model outputs back to local samples and grammars, scores exact match against both the canonical target and the full set of licensed targets, and summarizes results by agreement condition, intended grammar size, and sentence depth.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyrootutils
from IPython.display import display

PROJECT_ROOT = pyrootutils.find_root(indicator=".project-root")
DATA_DIR = PROJECT_ROOT / "data"
BATCHES_DIR = PROJECT_ROOT / "batches"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
FIGURES_DIR = PROJECT_ROOT / "paper" / "figures"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
EXP_NAME = "agreement"
EXP_DATA_DIR = DATA_DIR / f"{EXP_NAME}_exp"
EXP_BATCH_DIR = BATCHES_DIR / f"{EXP_NAME}_exp_compact"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

GRAMMAR_LIST = EXP_DATA_DIR / f"{EXP_NAME}_grammars.txt"
assert GRAMMAR_LIST.exists(), GRAMMAR_LIST
assert EXP_DATA_DIR.exists(), EXP_DATA_DIR
assert EXP_BATCH_DIR.exists(), EXP_BATCH_DIR

## Define aesthetics

In [ ]:
import aesthetics as aes  # noqa: F401

sns = aes.sns
plt = aes.plt
mtick = aes.mtick

aes.PALETTE_METRICS

AGREEMENT_CONDITION_ORDER = [
    "NoAgr → NoAgr",
    "Agr → NoAgr",
    "Agr → Agr",
    "NoAgr → Agr",
]

PALETTE_AGREEMENT = aes.darken(
    {
        "NoAgr → NoAgr": sns.color_palette("Dark2", n_colors=4)[0],
        "Agr → NoAgr": sns.color_palette("Dark2", n_colors=4)[1],
        "Agr → Agr": sns.color_palette("Dark2", n_colors=4)[2],
        "NoAgr → Agr": sns.color_palette("Dark2", n_colors=4)[3],
    },
    by=0.25,
)

# Set to a concrete model name to force plotting that model.
# Leave as None to auto-select from available rows.
PLOT_MODEL = None

## Load grammars and samples

In [ ]:
with open(GRAMMAR_LIST) as f:
    grammar_ids = [line.strip() for line in f if line.strip()]


def load_grammar(grammar_id: str) -> dict:
    with open(EXP_DATA_DIR / f"grammar_{grammar_id}.json") as f:
        grammar = json.load(f)
    meta = grammar.get("agreement_metadata", {})
    a_enabled = bool(meta.get("a", {}).get("config", {}).get("enabled", False))
    b_enabled = bool(meta.get("b", {}).get("config", {}).get("enabled", False))
    grammar_size = 5 * len(grammar["a"]["verbs"])
    return {
        "grammar_name": grammar_id,
        "grammar_size": grammar_size,
        "n_words": grammar.get("n_words"),
        "n_rules": grammar.get("n_rules"),
        "agreement_enabled_a": a_enabled,
        "agreement_enabled_b": b_enabled,
        "agreement_condition": (
            f"{'Agr' if a_enabled else 'NoAgr'} → {'Agr' if b_enabled else 'NoAgr'}"
        ),
    }


def extract_json_field(line: str, start_marker: str, end_markers: list[str]):
    start = line.index(start_marker) + len(start_marker)
    end = min(
        line.index(marker, start) for marker in end_markers if marker in line[start:]
    )
    return json.loads(line[start:end])


def load_sample_manifest(grammar_id: str) -> pd.DataFrame:
    rows = []
    with open(EXP_DATA_DIR / f"samples_{grammar_id}.jsonl") as f:
        for sample_id, line in enumerate(f):
            has_features = '"subject_features":' in line
            rows.append(
                {
                    "grammar_name": grammar_id,
                    "sample_id": str(sample_id),
                    "left_phonetic": extract_json_field(
                        line, '"left_phonetic": ', [', "right":']
                    ),
                    "right_phonetic": extract_json_field(
                        line,
                        '"right_phonetic": ',
                        [', "possible_right":', ', "left_tree":'],
                    ),
                    "depth": extract_json_field(
                        line,
                        '"depth": ',
                        [', "subject_features":', ', "grammar_name":'],
                    ),
                    "subject_features": extract_json_field(
                        line,
                        '"subject_features": ',
                        [', "verb_features":'],
                    )
                    if has_features
                    else {},
                    "verb_features": extract_json_field(
                        line,
                        '"verb_features": ',
                        [', "possible_right":'],
                    )
                    if has_features
                    else {},
                }
            )
    return pd.DataFrame(rows)


grammars_df = pd.DataFrame([load_grammar(grammar_id) for grammar_id in grammar_ids])
samples_df = pd.concat(
    [load_sample_manifest(grammar_id) for grammar_id in grammar_ids], ignore_index=True
)
grammars_df["agreement_condition"] = pd.Categorical(
    grammars_df["agreement_condition"],
    categories=AGREEMENT_CONDITION_ORDER,
    ordered=True,
)
samples_df["input_length"] = samples_df["left_phonetic"].str.split().str.len()
samples_df["target_length"] = samples_df["right_phonetic"].str.split().str.len()
samples_df["input_length_quintile"] = pd.qcut(
    samples_df["input_length"],
    q=5,
    duplicates="drop",
)
samples_df["input_length_quintile_mid"] = samples_df["input_length_quintile"].apply(
    lambda x: (x.left + x.right) / 2 if pd.notna(x) else np.nan
)
samples_df["subject_person"] = samples_df["subject_features"].apply(
    lambda x: x.get("person") if isinstance(x, dict) else None
)
samples_df["subject_number"] = samples_df["subject_features"].apply(
    lambda x: x.get("number") if isinstance(x, dict) else None
)
samples_df = samples_df.merge(grammars_df, on="grammar_name", how="left")
samples_df["agreement_condition"] = pd.Categorical(
    samples_df["agreement_condition"],
    categories=AGREEMENT_CONDITION_ORDER,
    ordered=True,
)

print(f"Grammars: {len(grammars_df)}")
print(f"Samples: {len(samples_df)}")
display(
    grammars_df.groupby(["agreement_condition", "grammar_size"], observed=False)
    .size()
    .rename("n_grammars")
    .reset_index()
)
display(
    samples_df.groupby(["agreement_condition", "input_length_quintile"], observed=False)
    .size()
    .rename("n_samples")
    .reset_index()
)

In [ ]:
samples_df.groupby("agreement_condition").agg(
    samples=("sample_id", "size"),
    mean_input_length=("input_length", "mean"),
    mean_target_length=("target_length", "mean"),
).round(2).reset_index()

## Load available batch outputs

In [ ]:
CUSTOM_ID_RE = re.compile(
    r"^(?P<grammar_name>[0-9a-f]+)-(?P<input_hash>[0-9a-f]+)-sample-(?P<sample_id>\d+)$"
)
ANSWER_RE = re.compile(
    r"final\s*answer\s*(?::|-|—)?\s*(?:is\s*)?([^\n]+)", re.IGNORECASE | re.DOTALL
)


def fuzzy_model(model: str) -> str:
    return re.sub(r"-\d{4}-\d{2}-\d{2}$", "", model or "")


def extract_answer(model_response: str | None) -> str | None:
    if not model_response:
        return None
    matches = ANSWER_RE.findall(model_response)
    if not matches:
        return None
    answer = matches[-1].strip()
    answer = re.sub(r"[^\w\s]", "", answer, flags=re.UNICODE).strip()
    return answer or None


def parse_usage(body: dict) -> tuple[int | None, int | None, int | None]:
    usage = body.get("usage", {}) or {}
    if "prompt_tokens" in usage:
        return (
            usage.get("prompt_tokens"),
            usage.get("completion_tokens"),
            usage.get("total_tokens"),
        )
    return (
        usage.get("promptTokens"),
        usage.get("completionTokens"),
        usage.get("totalTokens"),
    )


def load_outputs(batch_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(batch_dir.glob("*_output.jsonl")):
        with open(path) as f:
            for line in f:
                item = json.loads(line)
                body = (item.get("response") or {}).get("body") or {}
                message = None
                choices = body.get("choices") or []
                if choices:
                    message = ((choices[0] or {}).get("message") or {}).get("content")
                prompt_tokens, completion_tokens, total_tokens = parse_usage(body)
                match = CUSTOM_ID_RE.match(item.get("custom_id", ""))
                if not match:
                    continue
                row = {
                    "batch_file": path.name,
                    "batch_id": path.name.replace("_output.jsonl", ""),
                    "custom_id": item.get("custom_id"),
                    "grammar_name": match.group("grammar_name"),
                    "input_hash": match.group("input_hash"),
                    "sample_id": match.group("sample_id"),
                    "model": body.get("model"),
                    "fuzzy_model": fuzzy_model(body.get("model", "")),
                    "model_response": message,
                    "model_answer": extract_answer(message),
                    "status_code": (item.get("response") or {}).get("status_code"),
                    "error": item.get("error"),
                    "prompt_tokens": prompt_tokens,
                    "completion_tokens": completion_tokens,
                    "total_tokens": total_tokens,
                }
                rows.append(row)
    return pd.DataFrame(rows)


outputs_df = load_outputs(EXP_BATCH_DIR)
print(f"Output rows loaded: {len(outputs_df)}")
if len(outputs_df):
    display(
        outputs_df.groupby(["batch_file", "fuzzy_model"])
        .size()
        .rename("n_rows")
        .reset_index()
    )
else:
    print("No batch output files found yet.")

## Merge outputs with samples

In [ ]:
def json_string_array_contains(array_text: str, value: str | None) -> bool:
    if not value:
        return False
    needle = json.dumps(value, ensure_ascii=False)
    return any(
        token in array_text
        for token in (
            f"[{needle}]",
            f"[{needle}, ",
            f", {needle}, ",
            f", {needle}]",
        )
    )


def count_json_string_array_items(array_text: str) -> int:
    array_text = array_text.strip()
    if array_text == "[]":
        return 0
    return array_text.count('", "') + 1


def extract_possible_right_phonetic_array(line: str) -> str:
    start_marker = '"possible_right_phonetic": '
    end_markers = [', "agreement_ok":', ', "left_tree":', ', "grammar_name":']
    start = line.index(start_marker) + len(start_marker)
    end = min(
        line.index(marker, start) for marker in end_markers if marker in line[start:]
    )
    return line[start:end]


def load_reference_stats_for_outputs(outputs_df: pd.DataFrame) -> pd.DataFrame:
    if outputs_df.empty:
        return pd.DataFrame(
            columns=[
                "grammar_name",
                "sample_id",
                "reference_count",
                "is_ambiguous",
                "exact_match_any",
            ]
        )

    outputs_df = outputs_df.copy()
    outputs_df["reference_count"] = np.nan
    outputs_df["is_ambiguous"] = False
    outputs_df["exact_match_any"] = False

    needed = {}
    for idx, row in outputs_df.iterrows():
        needed.setdefault(row["grammar_name"], {}).setdefault(
            row["sample_id"], []
        ).append(idx)

    for grammar_name, sample_map in needed.items():
        sample_path = EXP_DATA_DIR / f"samples_{grammar_name}.jsonl"
        with open(sample_path) as f:
            for sample_idx, line in enumerate(f):
                sample_id = str(sample_idx)
                if sample_id not in sample_map:
                    continue
                array_text = extract_possible_right_phonetic_array(line)
                ref_count = count_json_string_array_items(array_text)
                for row_idx in sample_map[sample_id]:
                    answer = outputs_df.at[row_idx, "model_answer"]
                    outputs_df.at[row_idx, "reference_count"] = ref_count
                    outputs_df.at[row_idx, "is_ambiguous"] = ref_count > 1
                    outputs_df.at[row_idx, "exact_match_any"] = (
                        json_string_array_contains(array_text, answer)
                    )

    return outputs_df


merged_df = (
    outputs_df.merge(
        samples_df,
        on=["grammar_name", "sample_id"],
        how="left",
        validate="many_to_one",
    )
    if len(outputs_df)
    else pd.DataFrame()
)

if len(merged_df):
    merged_df = merged_df.drop_duplicates(subset=["batch_id", "custom_id"]).copy()
    merged_df = load_reference_stats_for_outputs(merged_df)
    merged_df["has_answer"] = merged_df["model_answer"].notna()
    merged_df["exact_match_canonical"] = merged_df.apply(
        lambda row: row["model_answer"] == row["right_phonetic"]
        if row["model_answer"]
        else False,
        axis=1,
    )
    merged_df["length_delta"] = merged_df.apply(
        lambda row: len(row["model_answer"].split()) - row["target_length"]
        if row["model_answer"]
        else np.nan,
        axis=1,
    )
    merged_df["ambiguity_bonus"] = (
        merged_df["exact_match_any"] & ~merged_df["exact_match_canonical"]
    )
    completion_rate = len(merged_df) / len(samples_df)
    print(
        f"Completed rows: {len(merged_df)} / {len(samples_df)} ({completion_rate:.1%})"
    )
    display(merged_df.head())
else:
    print("No outputs merged yet.")

## Progress snapshot

In [ ]:
if len(merged_df):
    progress_by_cell = (
        merged_df.groupby(["agreement_condition", "grammar_size", "depth"])
        .size()
        .rename("completed")
        .reset_index()
    )
    target_by_cell = (
        samples_df.groupby(["agreement_condition", "grammar_size", "depth"])
        .size()
        .rename("expected")
        .reset_index()
    )
    progress_df = target_by_cell.merge(
        progress_by_cell,
        on=["agreement_condition", "grammar_size", "depth"],
        how="left",
    ).fillna({"completed": 0})
    progress_df["completed"] = progress_df["completed"].astype(int)
    progress_df["pct_complete"] = progress_df["completed"] / progress_df["expected"]
    display(
        progress_df.sort_values(["agreement_condition", "grammar_size", "depth"]).head(
            20
        )
    )
else:
    progress_df = pd.DataFrame()
    print("No progress to report yet.")

## Accuracy metrics

In [ ]:
if len(merged_df):
    from difflib import SequenceMatcher

    try:
        import evaluate
    except ImportError:
        evaluate = None

    try:
        import sacrebleu
    except ImportError:
        sacrebleu = None

    def bow_match(row) -> bool:
        if row["model_answer"] is None or row["right_phonetic"] is None:
            return False
        return sorted(row["model_answer"].split()) == sorted(
            row["right_phonetic"].split()
        )

    def edit_similarity(row) -> float:
        pred = row["model_answer"] or ""
        ref = row["right_phonetic"] or ""
        return SequenceMatcher(None, pred, ref).ratio()

    def bleu_score(row) -> float:
        if sacrebleu is None:
            return np.nan
        pred = row["model_answer"] or ""
        ref = row["right_phonetic"] or ""
        return sacrebleu.sentence_bleu(pred, [ref]).score / 100.0

    merged_df["bow_match"] = merged_df.apply(bow_match, axis=1)
    merged_df["edit_similarity"] = merged_df.apply(edit_similarity, axis=1)
    merged_df["bleu"] = merged_df.apply(bleu_score, axis=1)

    if evaluate is not None:
        chrf = evaluate.load("chrf")
        merged_df["chrF++"] = [
            chrf.compute(
                predictions=[pred or ""],
                references=[[ref or ""]],
                beta=2,
                word_order=2,
            )["score"]
            / 100.0
            for pred, ref in zip(merged_df["model_answer"], merged_df["right_phonetic"])
        ]
    else:
        merged_df["chrF++"] = np.nan

    summary_by_model = (
        merged_df.groupby(["fuzzy_model"])
        .agg(
            rows=("custom_id", "size"),
            answered=("has_answer", "sum"),
            exact_match_canonical=("exact_match_canonical", "mean"),
            exact_match_any=("exact_match_any", "mean"),
            bow_match=("bow_match", "mean"),
            edit_similarity=("edit_similarity", "mean"),
            bleu=("bleu", "mean"),
            chrf_pp=("chrF++", "mean"),
            ambiguity_bonus=("ambiguity_bonus", "mean"),
            mean_prompt_tokens=("prompt_tokens", "mean"),
            mean_completion_tokens=("completion_tokens", "mean"),
        )
        .reset_index()
    )
    for col in [
        "exact_match_canonical",
        "exact_match_any",
        "bow_match",
        "edit_similarity",
        "bleu",
        "chrf_pp",
        "ambiguity_bonus",
    ]:
        summary_by_model[col] = (100 * summary_by_model[col]).round(2)
    summary_by_model
else:
    print("No outputs available for scoring yet.")

In [ ]:
PLOT_MODEL = "gpt-5"
# PLOT_MODEL = None

In [ ]:
if len(merged_df):
    PERF_METRICS = ["Exact Match"]
    selected_model = PLOT_MODEL
    print(f"Plotting model: {selected_model}")

    plot_source_df = merged_df[merged_df["fuzzy_model"] == selected_model].copy()
    plot_source_df["match_type"] = "Exact Match"
    plot_source_df["match_value"] = plot_source_df["exact_match_any"]

    plot_length_df = plot_source_df.copy()
    plot_size_df = plot_source_df.copy()
else:
    plot_length_df = pd.DataFrame()
    plot_size_df = pd.DataFrame()
    selected_model = None

## Plots

In [ ]:
if len(plot_length_df):
    fig = plt.figure(figsize=(aes.COLM_PAPER_WIDTH_IN, aes.FIG_HEIGHT_SINGLE_ROW_IN))
    grid = fig.add_gridspec(1, 2, wspace=0.1)

    with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
        ax = fig.add_subplot(grid[0, 0])
        sns.lineplot(
            data=plot_size_df,
            x="grammar_size",
            y="match_value",
            hue="agreement_condition",
            style="agreement_condition",
            hue_order=AGREEMENT_CONDITION_ORDER,
            style_order=AGREEMENT_CONDITION_ORDER,
            markers=True,
            markersize=8,
            palette=PALETTE_AGREEMENT,
            legend=True,
            errorbar="ci",
            ax=ax,
        )
        ax.set_title(None, fontweight="normal", loc="center")
        ax.set_xlabel("Grammar size")
        ax.set_ylabel("Exact match")
        ax.set_ylim(-0.05, 1.05)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.yaxis.grid(True, linestyle="--", alpha=0.5)
        ax.xaxis.set_major_formatter(aes.KMB_FORMATTER)

        ax = fig.add_subplot(grid[0, 1])
        sns.lineplot(
            data=plot_length_df,
            x="input_length_quintile_mid",
            y="match_value",
            hue="agreement_condition",
            style="agreement_condition",
            hue_order=AGREEMENT_CONDITION_ORDER,
            style_order=AGREEMENT_CONDITION_ORDER,
            markers=True,
            markersize=8,
            palette=PALETTE_AGREEMENT,
            legend=True,
            errorbar="ci",
            ax=ax,
        )
        ax.set_xlabel("Sentence length (binned into 5 quantiles)")
        ax.set_ylabel("")
        ax.set_ylim(-0.05, 1.05)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.yaxis.grid(True, linestyle="--", alpha=0.5)
        ax.yaxis.set_ticklabels([])

        left_legend = fig.axes[0].get_legend()
        if left_legend is not None:
            left_legend.remove()

        right_legend = fig.axes[1].get_legend()
        if right_legend is not None:
            right_legend.set_title("")
            sns.move_legend(
                fig.axes[1], "upper left", bbox_to_anchor=(1.02, 1.0), frameon=False
            )

    plt.subplots_adjust(left=0, bottom=0, right=1, top=1)
    aes.save_figure(FIGURES_DIR / f"{selected_model}_agreement", fig=fig)
    plt.show()
else:
    print("No scored outputs available yet for plotting.")

In [ ]:
EXPORT_DIR = NOTEBOOKS_DIR / "cache" / "combined-exp"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if len(plot_size_df) and len(plot_length_df):
    export_size_df = plot_size_df.copy()
    export_size_df["experiment"] = "agreement"
    export_size_df["panel"] = "grammar_size"
    export_size_df["selected_model"] = selected_model
    export_size_df.to_csv(EXPORT_DIR / "agreement_grammar_size.csv", index=False)

    export_length_df = plot_length_df.copy()
    export_length_df["experiment"] = "agreement"
    export_length_df["panel"] = "string_length"
    export_length_df["selected_model"] = selected_model
    export_length_df.to_csv(EXPORT_DIR / "agreement_string_length.csv", index=False)

    print(f"Saved combined-exp inputs to {EXPORT_DIR}")
else:
    print("No agreement plot data available to export.")